# More Complex GPU-Capable Example: Reservoir QUBO + QAOA

This is the **transpiled fixed version** of the GPU-capable reservoir QUBO + QAOA notebook.

The previous GPU version could fail with:

```text
AerError: 'unknown instruction: QAOA'
```

That happens when the QAOA ansatz reaches Aer with a high-level `QAOA` circuit instruction that has not been decomposed into backend-supported gates.

This version fixes that by:

1. creating an Aer GPU backend when available,
2. wrapping it with `BackendSamplerV2`,
3. creating a Qiskit preset pass manager for that backend,
4. passing that transpiler into `QAOA`.

So QAOA circuits are transpiled/decomposed before they reach Aer.

Problem size:

\[
T=6
\]

with 3 release levels:

\[
u(t)\in\{-\Delta u,0,\Delta u\}
\]

so the problem uses:

\[
3T=18
\]

binary variables / qubits.

## Expected behavior

The toy hydrology instance is designed to make the optimizer redistribute releases.

We use:

\[
R_{obs}=[100,100,100,100,100,100]
\]

\[
\Delta S^{obs}=[-70,60,-40,50,-30,30]
\]

\[
S_0=250,\qquad S_{min}=250.
\]

Since:

\[
\Delta u=0.25\cdot \text{median}(R_{obs})=25,
\]

the allowed release adjustments are:

\[
u(t)\in\{-25,0,25\}.
\]

A good expected schedule is:

\[
\text{levels}=[-1,-1,0,0,1,1]
\]

or:

\[
u=[-25,-25,0,0,25,25].
\]

This means the policy reduces release early when storage is stressed, then compensates with larger releases later.

## 1. Install dependencies

In [1]:
# In Google Colab:
# Runtime > Change runtime type > Hardware accelerator > T4 GPU

# CPU fallback installation:
# !pip install -q qiskit qiskit-aer qiskit-algorithms qiskit-optimization scipy numpy matplotlib

# GPU Aer installation:
# Try this in a Linux CUDA environment such as Colab T4.
# If it fails, use the CPU fallback above.
!pip install -q qiskit qiskit-aer-gpu qiskit-algorithms qiskit-optimization scipy numpy matplotlib

## 2. Imports and GPU-capable sampler setup

In [2]:
import itertools
import warnings
import subprocess
import shutil
import time
import inspect

import numpy as np
import matplotlib.pyplot as plt

from scipy.sparse import SparseEfficiencyWarning
warnings.filterwarnings("ignore", category=SparseEfficiencyWarning)

from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA

def print_gpu_info():
    if shutil.which("nvidia-smi") is None:
        print("nvidia-smi not found. GPU may not be attached.")
        return False

    try:
        out = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total,memory.free",
                "--format=csv,noheader",
            ],
            text=True,
        )
        print("Detected GPU:")
        print(out)
        return True
    except Exception as exc:
        print("Could not query GPU:", repr(exc))
        return False

gpu_visible = print_gpu_info()

# ------------------------------------------------------------
# Modern Qiskit-compatible sampler setup
# ------------------------------------------------------------
# We try:
#
#   BackendSamplerV2 + AerSimulator GPU
#
# and create a preset pass manager for the chosen backend.
# The pass manager is important because it decomposes the high-level
# QAOA instruction before Aer sees the circuit.
# ------------------------------------------------------------

sampler = None
sampler_mode = None
backend_for_test = None
qaoa_transpiler = None

try:
    from qiskit_aer import AerSimulator
    from qiskit.primitives import BackendSamplerV2
    from qiskit.transpiler import generate_preset_pass_manager

    try:
        gpu_backend = AerSimulator(
            method="statevector",
            device="GPU",
            cuStateVec_enable=True,
        )

        sampler = BackendSamplerV2(
            backend=gpu_backend,
            options={
                "default_shots": 512,
                "seed_simulator": 123,
            },
        )

        qaoa_transpiler = generate_preset_pass_manager(
            optimization_level=1,
            backend=gpu_backend,
            seed_transpiler=123,
        )

        sampler_mode = "BackendSamplerV2 + AerSimulator GPU + preset pass manager"
        backend_for_test = gpu_backend
        print("Using:", sampler_mode)

    except Exception as exc_gpu:
        print("GPU Aer setup failed:", repr(exc_gpu))

        cpu_backend = AerSimulator(method="statevector", device="CPU")

        sampler = BackendSamplerV2(
            backend=cpu_backend,
            options={
                "default_shots": 512,
                "seed_simulator": 123,
            },
        )

        qaoa_transpiler = generate_preset_pass_manager(
            optimization_level=1,
            backend=cpu_backend,
            seed_transpiler=123,
        )

        sampler_mode = "BackendSamplerV2 + AerSimulator CPU + preset pass manager"
        backend_for_test = cpu_backend
        print("Using:", sampler_mode)

except Exception as exc_aer:
    print("Aer/BackendSamplerV2 unavailable:", repr(exc_aer))

    from qiskit.primitives import StatevectorSampler
    sampler = StatevectorSampler(default_shots=512, seed=123)
    qaoa_transpiler = None

    sampler_mode = "StatevectorSampler V2 fallback, no Aer transpiler needed"
    print("Using:", sampler_mode)

print("Final sampler mode:", sampler_mode)

# Show QAOA signature so you can confirm whether the installed version supports transpiler.
print("QAOA signature:")
print(inspect.signature(QAOA))

Detected GPU:
Tesla T4, 15360 MiB, 14912 MiB

Aer/BackendSamplerV2 unavailable: ImportError("cannot import name 'convert_to_target' from 'qiskit.providers' (/opt/qcentroid-venv/lib/python3.12/site-packages/qiskit/providers/__init__.py)")
Using: StatevectorSampler V2 fallback, no Aer transpiler needed
Final sampler mode: StatevectorSampler V2 fallback, no Aer transpiler needed
QAOA signature:
(sampler: 'BaseSamplerV2', optimizer: 'Optimizer | Minimizer', *, reps: 'int' = 1, initial_state: 'QuantumCircuit | None' = None, mixer: 'QuantumCircuit | BaseOperator' = None, initial_point: 'np.ndarray | None' = None, aggregation: 'float | Callable[[list[float]], float] | None' = None, callback: 'Callable[[int, np.ndarray, float, dict[str, Any]], None] | None' = None, transpiler: 'Transpiler | None' = None, transpiler_options: 'dict[str, Any] | None' = None) -> 'None'


## 2.1 Optional GPU backend test

In [3]:
try:
    from qiskit import QuantumCircuit

    if backend_for_test is None:
        print("No Aer backend available for GPU/CPU backend test.")
    else:
        qc = QuantumCircuit(5)
        qc.h(range(5))
        qc.cx(0, 1)
        qc.cx(1, 2)
        qc.cx(2, 3)
        qc.cx(3, 4)
        qc.measure_all()

        test_result = backend_for_test.run(qc, shots=128).result()
        print("Aer backend test succeeded.")
        print(test_result.get_counts())
        print("Backend options:")
        print(backend_for_test.options)
except Exception as exc:
    print("Aer backend test failed, but the notebook can still continue with fallback.")
    print("Reason:", repr(exc))

No Aer backend available for GPU/CPU backend test.


## 3. QUBO helper functions

In [4]:
def add_quadratic(Q, i, j, value):
    """
    Add value * x_i * x_j using upper-triangular QUBO convention.
    """
    if i == j:
        Q[i, i] += value
    else:
        a, b = min(i, j), max(i, j)
        Q[a, b] += value


def add_squared_expression(Q, constant, coeffs, penalty):
    """
    Add:

        penalty * (constant + sum_i coeffs[i] x_i)^2

    to the QUBO matrix Q.

    Since x_i is binary, x_i^2 = x_i.

    Diagonal:
        penalty * (r_i^2 + 2 constant r_i)

    Off-diagonal:
        penalty * 2 r_i r_j
    """
    items = list(coeffs.items())

    for i, ri in items:
        Q[i, i] += penalty * (ri**2 + 2.0 * constant * ri)

    for p in range(len(items)):
        i, ri = items[p]
        for q in range(p + 1, len(items)):
            j, rj = items[q]
            add_quadratic(Q, i, j, penalty * 2.0 * ri * rj)


def qubo_energy(x, Q):
    return float(x @ Q @ x)

## 4. Build the \(T=6\), 18-qubit reservoir QUBO

In [5]:
def make_index(T):
    """
    3 release levels:

        a in {-1, 0, 1}

    Variable order per time step:

        x_{t,-1}, x_{t,0}, x_{t,1}
    """
    levels = [-1, 0, 1]
    index = {}
    counter = 0

    for t in range(T):
        for a in levels:
            index[(t, a)] = counter
            counter += 1

    return index, levels, counter


def build_complex_gpu_qubo(
    R_obs,
    dS_obs,
    S0,
    Smin,
    lambda_onehot=40.0,
    lambda_R=40.0,
    lambda_bal=0.002,
    crit_scale=100.0,
    include_crit=True,
    include_smooth=True,
    include_dev=True,
):
    """
    Build a QUBO with 3 release levels:

        u(t) in {-Delta u, 0, Delta u}

    Number of variables / qubits = 3T.

    crit_scale is intentionally larger than the official benchmark weight
    so that this small educational example produces visible storage-protection behavior.
    """
    R_obs = np.asarray(R_obs, dtype=float)
    dS_obs = np.asarray(dS_obs, dtype=float)

    T = len(R_obs)
    assert len(dS_obs) == T

    index, levels, n = make_index(T)
    Q = np.zeros((n, n), dtype=float)

    delta_u = 0.25 * np.median(R_obs)
    umax = delta_u

    w1 = crit_scale / ((T + 1) * Smin**2)
    w2 = 0.1 / (T * umax**2)
    w3 = 0.1 / ((T - 1) * (2 * umax)**2) if T > 1 else 0.0

    # Cdev = sum_t u(t)^2
    if include_dev:
        for t in range(T):
            coeffs = {index[(t, a)]: delta_u * a for a in levels}
            add_squared_expression(Q, constant=0.0, coeffs=coeffs, penalty=w2)

    # Csmooth = sum_{t=1}^{T-1} [u(t)-u(t-1)]^2
    if include_smooth:
        for t in range(1, T):
            coeffs = {}

            for a in levels:
                coeffs[index[(t, a)]] = coeffs.get(index[(t, a)], 0.0) + delta_u * a
                coeffs[index[(t - 1, a)]] = coeffs.get(index[(t - 1, a)], 0.0) - delta_u * a

            add_squared_expression(Q, constant=0.0, coeffs=coeffs, penalty=w3)

    # Approximate Ccrit without max in the QUBO:
    #
    # Smin - Sopt(t) = A_t + sum_{tau<t} u(tau)
    # A_t = Smin - S0 - sum_{tau<t} dS_obs(tau)
    if include_crit:
        for t in range(T + 1):
            A_t = Smin - S0 - np.sum(dS_obs[:t])

            coeffs = {}
            for tau in range(t):
                for a in levels:
                    i = index[(tau, a)]
                    coeffs[i] = coeffs.get(i, 0.0) + delta_u * a

            add_squared_expression(Q, constant=A_t, coeffs=coeffs, penalty=w1)

    # One-hot constraint per time step
    for t in range(T):
        coeffs = {index[(t, a)]: 1.0 for a in levels}
        add_squared_expression(Q, constant=-1.0, coeffs=coeffs, penalty=lambda_onehot)

    # R(t) >= 0 simple diagonal penalty
    for t in range(T):
        for a in levels:
            u_value = delta_u * a
            if R_obs[t] + u_value < 0:
                i = index[(t, a)]
                Q[i, i] += lambda_R

    # Simplified balance:
    #
    # Pbal = lambda_bal * (sum_t u(t))^2
    coeffs = {}
    for t in range(T):
        for a in levels:
            coeffs[index[(t, a)]] = delta_u * a

    add_squared_expression(Q, constant=0.0, coeffs=coeffs, penalty=lambda_bal)

    metadata = {
        "T": T,
        "levels": levels,
        "index": index,
        "num_binary_variables": n,
        "delta_u": delta_u,
        "umax": umax,
        "w1": w1,
        "w2": w2,
        "w3": w3,
        "crit_scale": crit_scale,
    }

    return Q, metadata

## 5. Convert QUBO to Qiskit `QuadraticProgram`

In [6]:
def qubo_matrix_to_quadratic_program(Q):
    """
    Convert an upper-triangular QUBO matrix to a Qiskit QuadraticProgram.
    """
    from qiskit_optimization import QuadraticProgram

    n = Q.shape[0]
    qp = QuadraticProgram("complex_gpu_reservoir_qubo")

    for i in range(n):
        qp.binary_var(name=f"x_{i}")

    linear = {}
    quadratic = {}

    for i in range(n):
        if abs(Q[i, i]) > 1e-12:
            linear[f"x_{i}"] = Q[i, i]

    for i in range(n):
        for j in range(i + 1, n):
            if abs(Q[i, j]) > 1e-12:
                quadratic[(f"x_{i}", f"x_{j}")] = Q[i, j]

    qp.minimize(linear=linear, quadratic=quadratic)
    return qp

## 6. Decode and evaluate solutions

In [7]:
def levels_to_binary(selected_levels, metadata):
    T = metadata["T"]
    index = metadata["index"]
    n = metadata["num_binary_variables"]

    x = np.zeros(n)

    for t, a in enumerate(selected_levels):
        x[index[(t, a)]] = 1

    return x


def decode_solution(result_x, metadata):
    T = metadata["T"]
    levels = metadata["levels"]
    index = metadata["index"]
    delta_u = metadata["delta_u"]

    selected_levels = []
    u = []

    for t in range(T):
        active = []

        for a in levels:
            i = index[(t, a)]
            if int(round(result_x[i])) == 1:
                active.append(a)

        if len(active) == 1:
            chosen = active[0]
        else:
            chosen = max(levels, key=lambda a: result_x[index[(t, a)]])

        selected_levels.append(chosen)
        u.append(delta_u * chosen)

    return np.array(selected_levels), np.array(u)


def compute_storage_and_score(u, R_obs, dS_obs, S0, Smin, crit_scale=100.0):
    R_obs = np.asarray(R_obs, dtype=float)
    dS_obs = np.asarray(dS_obs, dtype=float)
    u = np.asarray(u, dtype=float)

    T = len(u)
    Ropt = R_obs + u

    Sopt = np.zeros(T + 1)
    Sopt[0] = S0

    for t in range(T):
        Sopt[t + 1] = Sopt[t] + dS_obs[t] - u[t]

    delta_u = 0.25 * np.median(R_obs)
    umax = delta_u

    w1 = crit_scale / ((T + 1) * Smin**2)
    w2 = 0.1 / (T * umax**2)
    w3 = 0.1 / ((T - 1) * (2 * umax)**2) if T > 1 else 0.0

    Ccrit = np.sum(np.maximum(0.0, Smin - Sopt) ** 2)
    Cdev = np.sum(u**2)
    Csmooth = np.sum((u[1:] - u[:-1]) ** 2) if T > 1 else 0.0
    SRS = -(w1 * Ccrit + w2 * Cdev + w3 * Csmooth)

    return {
        "Ropt": Ropt,
        "Sopt": Sopt,
        "Ccrit": Ccrit,
        "Cdev": Cdev,
        "Csmooth": Csmooth,
        "SRS": SRS,
    }

## 7. Brute-force verification

In [8]:
def brute_force_onehot(Q, metadata):
    """
    Brute-force all valid one-hot release schedules.

    For T=6 and 3 levels, this checks:

        3^6 = 729

    schedules, which is still fast and useful for verifying the QUBO.
    """
    T = metadata["T"]
    levels = metadata["levels"]

    best = None
    rows = []

    for selected in itertools.product(levels, repeat=T):
        x = levels_to_binary(selected, metadata)
        energy = qubo_energy(x, Q)

        rows.append((selected, energy))

        if best is None or energy < best[1]:
            best = (selected, energy, x)

    rows = sorted(rows, key=lambda z: z[1])
    return best, rows

## 7.1 Why the transpiler is needed

The Aer backend can only execute instructions it knows how to simulate. In some Qiskit versions, the QAOA ansatz reaches the backend as a high-level instruction named `QAOA`.

If that high-level instruction is not decomposed first, Aer raises:

```text
AerError: 'unknown instruction: QAOA'
```

To avoid this, the QAOA solver below passes a preset pass manager into the `QAOA` constructor. That pass manager transpiles/decomposes the circuit before sampling.

## 8. QAOA solver

In [9]:
def solve_with_qaoa(Q, reps=1, maxiter=30):
    """
    Solve the QUBO with QAOA.

    This version passes a transpiler/pass manager into QAOA when using Aer.
    That prevents:

        AerError: 'unknown instruction: QAOA'

    because the high-level QAOA instruction is decomposed before reaching Aer.
    """
    from qiskit_algorithms import QAOA
    from qiskit_algorithms.optimizers import COBYLA
    from qiskit_optimization.algorithms import MinimumEigenOptimizer

    if "sampler" not in globals():
        from qiskit.primitives import StatevectorSampler
        globals()["sampler"] = StatevectorSampler(default_shots=512, seed=123)
        globals()["sampler_mode"] = "StatevectorSampler V2 fallback"
        globals()["qaoa_transpiler"] = None

    qp = qubo_matrix_to_quadratic_program(Q)

    qaoa_kwargs = {
        "sampler": sampler,
        "optimizer": COBYLA(maxiter=maxiter),
        "reps": reps,
    }

    # Newer qiskit_algorithms.QAOA supports:
    #
    #   transpiler=...
    #   transpiler_options=...
    #
    # If your version does not support it, this will fall back automatically.
    if globals().get("qaoa_transpiler", None) is not None:
        qaoa_kwargs["transpiler"] = qaoa_transpiler
        qaoa_kwargs["transpiler_options"] = {}

    try:
        qaoa = QAOA(**qaoa_kwargs)
    except TypeError as exc:
        print("QAOA did not accept transpiler arguments in this environment.")
        print("Falling back to StatevectorSampler without Aer backend.")
        print("Original TypeError:", repr(exc))

        from qiskit.primitives import StatevectorSampler
        fallback_sampler = StatevectorSampler(default_shots=512, seed=123)

        qaoa = QAOA(
            sampler=fallback_sampler,
            optimizer=COBYLA(maxiter=maxiter),
            reps=reps,
        )

    optimizer = MinimumEigenOptimizer(qaoa)
    result = optimizer.solve(qp)

    return result, qp

## 9. Define the more complex \(T=6\) instance

In [10]:
T = 6

R_obs = np.array([100.0, 100.0, 100.0, 100.0, 100.0, 100.0])
dS_obs = np.array([-70.0, 60.0, -40.0, 50.0, -30.0, 30.0])

Smax = 1000.0
Smin = 0.25 * Smax
S0 = 250.0

Q, metadata = build_complex_gpu_qubo(
    R_obs=R_obs,
    dS_obs=dS_obs,
    S0=S0,
    Smin=Smin,
    lambda_onehot=40.0,
    lambda_R=40.0,
    lambda_bal=0.002,
    crit_scale=100.0,
)

print("QUBO shape:", Q.shape)
print("Number of variables / qubits:", metadata["num_binary_variables"])
print("Release levels:", metadata["levels"])
print("delta_u:", metadata["delta_u"])
print("umax:", metadata["umax"])
print("w1:", metadata["w1"])
print("w2:", metadata["w2"])
print("w3:", metadata["w3"])
print("Sampler mode:", sampler_mode)

QUBO shape: (18, 18)
Number of variables / qubits: 18
Release levels: [-1, 0, 1]
delta_u: 25.0
umax: 25.0
w1: 0.00022857142857142857
w2: 2.6666666666666667e-05
w3: 8e-06
Sampler mode: StatevectorSampler V2 fallback, no Aer transpiler needed


## 10. Verify optimum by brute force

In [11]:
start = time.time()
best, rows = brute_force_onehot(Q, metadata)
elapsed = time.time() - start

print("Brute force elapsed seconds:", round(elapsed, 4))
print("Best valid one-hot schedule:")
print("levels =", best[0])
print("QUBO energy =", best[1])

print("\\nTop 10 valid schedules:")
for selected, energy in rows[:10]:
    selected_arr = np.array(selected)
    u = selected_arr * metadata["delta_u"]

    metrics_tmp = compute_storage_and_score(
        u=u,
        R_obs=R_obs,
        dS_obs=dS_obs,
        S0=S0,
        Smin=Smin,
        crit_scale=metadata["crit_scale"],
    )

    print(
        "levels =", tuple(selected_arr),
        "u =", u,
        "energy =", round(energy, 6),
        "Sopt =", metrics_tmp["Sopt"],
        "SRS =", round(metrics_tmp["SRS"], 6),
    )

Brute force elapsed seconds: 0.0047
Best valid one-hot schedule:
levels = (-1, 0, -1, 1, 0, 1)
QUBO energy = -241.15047619047618
\nTop 10 valid schedules:
levels = (np.int64(-1), np.int64(0), np.int64(-1), np.int64(1), np.int64(0), np.int64(1)) u = [-25.   0. -25.  25.   0.  25.] energy = -241.150476 Sopt = [250. 205. 265. 250. 275. 245. 250.] SRS = -0.575238
levels = (np.int64(-1), np.int64(0), np.int64(0), np.int64(1), np.int64(-1), np.int64(1)) u = [-25.   0.   0.  25. -25.  25.] energy = -241.140476 Sopt = [250. 205. 265. 225. 250. 245. 250.] SRS = -0.728095
levels = (np.int64(-1), np.int64(1), np.int64(-1), np.int64(1), np.int64(-1), np.int64(1)) u = [-25.  25. -25.  25. -25.  25.] energy = -241.085714 Sopt = [250. 205. 240. 225. 250. 245. 250.] SRS = -0.834286
levels = (np.int64(-1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1)) u = [-25.   0.   0.   0.   0.  25.] energy = -241.070952 Sopt = [250. 205. 265. 225. 275. 245. 250.] SRS = -0.654762
levels = (np.int6

## 11. Run GPU-capable QAOA

In [ ]:
# Start with reps=1, maxiter=30.
# If it is still slow, try maxiter=10.
# If it is fast but unstable, try maxiter=60 or reps=2.

start = time.time()

result, qp = solve_with_qaoa(Q, reps=1, maxiter=5)

elapsed = time.time() - start

print("QAOA elapsed seconds:", round(elapsed, 4))
print(result)

## 12. Decode QAOA solution

In [ ]:
selected_levels, uopt = decode_solution(result.x, metadata)

print("Selected levels from QAOA:")
print(selected_levels)

print("\\nOptimized u(t):")
print(uopt)

metrics = compute_storage_and_score(
    u=uopt,
    R_obs=R_obs,
    dS_obs=dS_obs,
    S0=S0,
    Smin=Smin,
    crit_scale=metadata["crit_scale"],
)

print("\\nRopt:")
print(metrics["Ropt"])

print("\\nSopt:")
print(metrics["Sopt"])

print("\\nScore terms:")
for key in ["Ccrit", "Cdev", "Csmooth", "SRS"]:
    print(key, "=", metrics[key])

print("\\nRaw binary vector:")
print(result.x)

## 13. Compare QAOA, brute force, and doing nothing

In [ ]:
# Brute-force best
bf_levels = np.array(best[0])
bf_u = bf_levels * metadata["delta_u"]
bf_metrics = compute_storage_and_score(
    u=bf_u,
    R_obs=R_obs,
    dS_obs=dS_obs,
    S0=S0,
    Smin=Smin,
    crit_scale=metadata["crit_scale"],
)

# Doing nothing
u_zero = np.zeros(T)
zero_metrics = compute_storage_and_score(
    u=u_zero,
    R_obs=R_obs,
    dS_obs=dS_obs,
    S0=S0,
    Smin=Smin,
    crit_scale=metadata["crit_scale"],
)

print("Doing nothing:")
print("u =", u_zero)
print("Sopt =", zero_metrics["Sopt"])
print("Ccrit =", zero_metrics["Ccrit"])
print("SRS =", zero_metrics["SRS"])

print("\\nBrute-force best:")
print("levels =", bf_levels)
print("u =", bf_u)
print("Sopt =", bf_metrics["Sopt"])
print("Ccrit =", bf_metrics["Ccrit"])
print("SRS =", bf_metrics["SRS"])

print("\\nQAOA:")
print("levels =", selected_levels)
print("u =", uopt)
print("Sopt =", metrics["Sopt"])
print("Ccrit =", metrics["Ccrit"])
print("SRS =", metrics["SRS"])

## 14. Plot storage and release

In [ ]:
time_storage = np.arange(T + 1)
time_release = np.arange(T)

plt.figure()
plt.plot(time_storage, zero_metrics["Sopt"], marker="o", label="u=0")
plt.plot(time_storage, bf_metrics["Sopt"], marker="o", label="Brute force best")
plt.plot(time_storage, metrics["Sopt"], marker="o", label="QAOA")
plt.axhline(Smin, linestyle="--", label="Smin")
plt.xlabel("Time")
plt.ylabel("Storage")
plt.title("Storage trajectory comparison")
plt.legend()
plt.show()

plt.figure()
width = 0.25
plt.bar(time_release - width, u_zero, width=width, label="u=0")
plt.bar(time_release, bf_u, width=width, label="Brute force best")
plt.bar(time_release + width, uopt, width=width, label="QAOA")
plt.axhline(0.0)
plt.xlabel("Time")
plt.ylabel("u(t)")
plt.title("Release adjustment comparison")
plt.legend()
plt.show()

plt.figure()
plt.plot(time_release, R_obs, marker="o", label="Robs")
plt.plot(time_release, bf_metrics["Ropt"], marker="o", label="Ropt brute force")
plt.plot(time_release, metrics["Ropt"], marker="o", label="Ropt QAOA")
plt.xlabel("Time")
plt.ylabel("Release")
plt.title("Observed vs optimized release")
plt.legend()
plt.show()

## 15. Optional: make it larger

You can try \(T=8\), which gives:

\[
3T=24
\]

qubits and:

\[
3^8=6561
\]

valid one-hot schedules for brute-force verification.

Replace the data cell with:

```python
T = 8
R_obs = np.ones(T) * 100.0
dS_obs = np.array([-70.0, 60.0, -40.0, 50.0, -30.0, 30.0, -50.0, 60.0])
Smax = 1000.0
Smin = 0.25 * Smax
S0 = 250.0
```

Then rebuild `Q` and rerun the brute-force and QAOA cells.

For \(T=8\), if QAOA is too slow, use:

```python
result, qp = solve_with_qaoa(Q, reps=1, maxiter=10)
```

or skip QAOA first and use brute force to validate the QUBO.

## 16. Troubleshooting

### If you still see `AerError: 'unknown instruction: QAOA'`

Restart the runtime and run all cells from the top. The fix depends on the import cell creating:

```python
qaoa_transpiler = generate_preset_pass_manager(...)
```

and the solver passing it into:

```python
QAOA(..., transpiler=qaoa_transpiler)
```

### If your installed QAOA does not support `transpiler=...`

The solver automatically falls back to `StatevectorSampler`. This is not GPU-backed, but it avoids the Aer instruction error and lets the notebook complete.

### If GPU is slower than CPU

For 18 qubits, GPU overhead can dominate. The GPU path becomes more useful for larger circuits. For quick validation, rely on the brute-force cell first.